# Multi-Object Detection & Persistent ID Tracking
### Interactive Notebook

This notebook lets you:
1. Run detection on a single test image
2. Run the full tracking pipeline on a video
3. Visualise results (heatmap, count-over-time)


In [ ]:
import sys
sys.path.insert(0, 'src')

import cv2
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Image as IPImage

print('✅ Imports OK')

## 1. Single-Image Detection Test

In [ ]:
from detector import Detector
from visualize import draw_tracks

# Load image
IMAGE_PATH = 'data/input/sample_frame.jpg'  # change path as needed
frame = cv2.imread(IMAGE_PATH)
if frame is None:
    print('⚠️  Image not found – using a blank canvas for demo')
    frame = np.zeros((480, 640, 3), dtype=np.uint8)

detector = Detector(model_path='src/yolov8n.pt')
detections = detector.detect(frame)

print(f'Detections: {len(detections)}')
for bbox, conf, label in detections:
    x1,y1,x2,y2 = [int(v) for v in bbox]
    cv2.rectangle(frame, (x1,y1), (x2,y2), (0,255,0), 2)
    cv2.putText(frame, f'{label} {conf:.2f}', (x1, y1-10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)

# Show inline
rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
plt.figure(figsize=(12, 7))
plt.imshow(rgb)
plt.axis('off')
plt.title('Detection Result')
plt.tight_layout()
plt.show()

## 2. Run Full Pipeline

In [ ]:
# Run main.py from within the notebook
import subprocess
result = subprocess.run(
    ['python', 'src/main.py',
     '--input',      'data/input/video.mp4',
     '--output',     'data/output/output.mp4',
     '--no-display',
     '--frame-skip', '2'],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

## 3. View Heatmap

In [ ]:
heatmap = cv2.imread('data/output/heatmap.png')
if heatmap is not None:
    rgb = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(14, 8))
    plt.imshow(rgb)
    plt.axis('off')
    plt.title('Movement Heatmap (foot positions)')
    plt.tight_layout()
    plt.show()
else:
    print('Run the pipeline first.')

## 4. Person Count Over Time

In [ ]:
import csv

frames, counts = [], []
try:
    with open('data/output/count_over_time.csv') as f:
        for row in csv.DictReader(f):
            frames.append(int(row['frame']))
            counts.append(int(row['person_count']))

    plt.figure(figsize=(14, 4))
    plt.plot(frames, counts, linewidth=1.2, color='#2196F3')
    plt.fill_between(frames, counts, alpha=0.15, color='#2196F3')
    plt.xlabel('Frame')
    plt.ylabel('People detected')
    plt.title('Person Count Over Time')
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
    print(f'Max: {max(counts)}, Min: {min(counts)}, Avg: {sum(counts)/len(counts):.1f}')
except FileNotFoundError:
    print('Run the pipeline first.')